In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, make_scorer, f1_score,
    precision_score, recall_score, roc_curve
)
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from scipy.stats import randint, uniform


import warnings
warnings.filterwarnings('ignore')

In [2]:
import Models_for_Control

final_matrix = pd.read_parquet(r"MODEL_READY_MATRIX.parquet")
# make a new balanced dataset by randomly sampling controls to match the number of disease patients
RANDOM_SEEDS = [42, 69, 420, 80085]

disease = final_matrix[final_matrix['LABEL'] == 1]
control = final_matrix[final_matrix['LABEL'] == 0]

n_disease = len(disease)
n_control = len(control)

print(f"Disease patients: {n_disease}")
print(f"Control patients: {n_control}")
print(f"Imbalance ratio: 1:{n_control // n_disease}")

results = []

for RANDOM_SEED in RANDOM_SEEDS:
    # random sample of controls, same size as disease cohort
    control_sampled = control.sample(n=n_disease, random_state=RANDOM_SEED)

    # concat and shuffle
    balanced_matrix = pd.concat([disease, control_sampled], ignore_index=True)
    balanced_matrix = balanced_matrix.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    #print(f"\nBalanced matrix shape: {balanced_matrix.shape}")
    #print(f"  Label=1 (Disease): {(balanced_matrix['LABEL']==1).sum()}")
    #print(f"  Label=0 (Control): {(balanced_matrix['LABEL']==0).sum()}")

    # Train models on the balanced matrix
    print(f"\n[Seed {RANDOM_SEED}] Training models on balanced dataset...")
    lasso_cv, lasso_importance_df = Models_for_Control.LR(balanced_matrix)
    rf_cv, rf_tuned_cv, rf_tuned_importance_df = Models_for_Control.RF(balanced_matrix)
    for name, cv_result in [('lasso', lasso_cv), ('rf_tuned', rf_tuned_cv)]:
        row = {'seed': RANDOM_SEED, 'model': name}
        for metric in ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']:
            scores = cv_result[f'test_{metric}']
            row[f'{metric}_mean'] = scores.mean()
            row[f'{metric}_std'] = scores.std()
        results.append(row)

    # stash importances too
    lasso_importance_df.to_csv(f'lasso_importance_seed{RANDOM_SEED}.csv', index=False)
    rf_tuned_importance_df.to_csv(f'rf_tuned_importance_seed{RANDOM_SEED}.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv('control_selection_comparison.csv', index=False)
print(results_df)

Disease patients: 76
Control patients: 46177
Imbalance ratio: 1:607

[Seed 42] Training models on balanced dataset...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Seed 69] Training models on balanced dataset...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Seed 420] Training models on balanced dataset...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Seed 80085] Training models on balanced dataset...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
    seed     model  accuracy_mean  accuracy_std   f1_mean    f1_std  \
0     42     lasso       0.670968      0.041142  0.607576  0.098357   
1     42  rf_tuned       0.715914      0.112299  0.745556  0.112897   
2     69     lasso       0.637419      0.059479  0.599540  0.084092   
3     69  rf_tuned       0.729677      0.046710  0.745039  0.054010   
4    420     lasso       0.716774      0.058484  0.688308  0.070112   
5    420  rf_tuned       0.749892      0.053916  0.768

In [5]:
# aggregate across seeds per model
metrics = ['accuracy_mean', 'f1_mean', 'precision_mean', 'recall_mean', 'roc_auc_mean']
summary = results_df.groupby('model')[metrics].agg(['mean', 'std'])
summary.columns = [f'{m}_{s}' for m, s in summary.columns]
summary.to_csv('control_selection_summary.csv')
print(summary.round(4))

          accuracy_mean_mean  accuracy_mean_std  f1_mean_mean  f1_mean_std  \
model                                                                        
lasso                 0.6708             0.0336        0.6230       0.0438   
rf_tuned              0.7231             0.0223        0.7451       0.0193   

          precision_mean_mean  precision_mean_std  recall_mean_mean  \
model                                                                 
lasso                  0.7327              0.0489            0.5650   
rf_tuned               0.6920              0.0226            0.8158   

          recall_mean_std  roc_auc_mean_mean  roc_auc_mean_std  
model                                                           
lasso              0.0456             0.7566            0.0079  
rf_tuned           0.0237             0.7773            0.0215  
